# LangGraph

In [ ]:
!pip install -U langgraph

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

In [ ]:
#define state
class State(TypedDict):
  name: str
  message: str

In [ ]:
#define node
def greeting_node(state: State):
  return {
      "message" : f"Hello {state['name']}"
  }

def farewell_node(state: State):
  return {
      "message" : f"Goodbye {state['name']}"
  }

In [ ]:
#create graph
graph = StateGraph(State)

#add node
graph.add_node("greeting", greeting_node)
graph.add_node("farewell", farewell_node)

#add edge
graph.add_edge(START, "greeting")
graph.add_edge("greeting", "farewell")
graph.add_edge("farewell", END)

In [ ]:
#compile
app = graph.compile()
app

In [ ]:
#run
Name = input("Enter name: ")

#single end node invoke access
# result = app.invoke({"name": Name})
# print(result)

#multi-node access
for res in app.stream({"name" : Name}):
  print(res)

## LLM

In [ ]:
!pip install langchain langchain-community langchain-google-genai

In [ ]:
from google.colab import userdata
import os

key = userdata.get("GEMINI_API_KEY")
os.environ["GOOGLE_API_KEY"] = key

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model = "gemini-3.5-flash-lite",
    temperature = 0
)

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

In [ ]:
class State(TypedDict):
    question: str
    answer: str

In [ ]:
def llm_node(state: State):
  response = llm.invoke(state["question"])
  return {"answer": response.content}

In [ ]:
graph = StateGraph(State)

graph.add_node("llm_call", llm_node)

graph.add_edge(START, "llm_call")
graph.add_edge("llm_call", END)

app = graph.compile()
app

In [ ]:
ques = input("Ask question: ")
result = app.invoke({"question": ques})
print(result["answer"])

### LLM and TOOLS

In [ ]:
from typing_extensions import TypedDict

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.tools import tool
from langchain_core.messages import ToolMessage
from langgraph.graph import StateGraph, START, END

In [ ]:
@tool
def add(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers."""
    return a * b

@tool
def divide(a: int, b: int) -> float:
    """Divide two numbers."""
    return a / b

In [ ]:
#tools
tools = [add, multiply, divide]
tools_by_name = {tool.name: tool
                 for tool in tools}

#llm
model = ChatGoogleGenerativeAI(
    model = "gemini-3.5-flash-lite",
    temperature = 0
)
model_with_tools = model.bind_tools(tools)

#state
class State(TypedDict):
  messages: list

In [ ]:
#llm node
def llm_call(state):
  response = model_with_tools.invoke(state["messages"])
  return {"messages": [response]}

#tool node
def tool_node(state):
  results = []
  last_msg = state["messages"][-1]
  for tool_call in last_msg.tool_calls:
    tool_name = tool_call["name"]
    tool_args = tool_call["args"]
    tool = tools_by_name[tool_name]
    result = tool.invoke(tool_args)
    results.append(
        ToolMessage(
            content = str(result),
            tool_call_id = tool_call["id"]))
    return {"messages": results}

#routing
def should_continue(state):
  last_msg = state["messages"][-1]
  if last_msg.tool_calls:
    return "tool_node"
  return "end"

In [ ]:
#build graph
builder = StateGraph(State)

builder.add_node("llm_call", llm_call)
builder.add_node("tool_node", tool_node)

builder.add_edge(START, "llm_call")
builder.add_conditional_edges("llm_call",
    should_continue,
    {"tool_node": "tool_node", "end": END}
)
builder.add_edge("tool_node","llm_call")

agent = builder.compile()

ques = input("Enter calculation: ")
res = agent.invoke({"messages": [{"role": "user", "content": ques}]})
print(res["messages"][-1].content)

Enter calculation: what is 4*6


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


ValueError: contents are required.